In [2]:
# CUDA used for parallel processing 

from numba import cuda
import numpy as np
@cuda.jit
def hello_kernel():
  tx=cuda.threadIdx.x
  bx=cuda.blockIdx.x
  bdim=cuda.blockDim.x
  gid=bx*bdim+tx
  print("hello from block",bx,"thread",tx,"globalid",gid)
threads_per_block = 4
blocks = 2
hello_kernel[blocks,threads_per_block]()
cuda.synchronize()

hello from block 0 thread 0 globalid 0
hello from block 0 thread 1 globalid 1
hello from block 0 thread 2 globalid 2
hello from block 0 thread 3 globalid 3
hello from block 1 thread 0 globalid 4
hello from block 1 thread 1 globalid 5
hello from block 1 thread 2 globalid 6
hello from block 1 thread 3 globalid 7


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [4]:
import numpy as np
from numba import cuda
@cuda.jit
def add_arrays(a, b, c):
    i = cuda.grid(1)        # get thread index
    if i < a.size:         # avoid out-of-bounds
        c[i] = a[i] + b[i]
a = np.array([1, 2, 3, 4, 5], dtype=np.float32)
b = np.array([10, 20, 30, 40, 50], dtype=np.float32)
c = np.zeros(5, dtype=np.float32)
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)
threads_per_block = 32
blocks = 1

add_arrays[blocks, threads_per_block](d_a, d_b, d_c)
result = d_c.copy_to_host()
print("Result:", result)


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


Result: [11. 22. 33. 44. 55.]


In [5]:
import numpy as np
from numba import cuda

# CUDA kernel to add two arrays
@cuda.jit
def add_arrays_gpu(a, b, result):
    idx = cuda.grid(1)  # get global thread index
    if idx < a.size:
        result[idx] = a[idx] + b[idx]

# Define arrays directly in the program
a = np.array([1, 2, 3, 4, 5], dtype=np.float32)
b = np.array([5, 4, 3, 2, 1], dtype=np.float32)
result = np.zeros(a.size, dtype=np.float32)

# CUDA configuration
threads_per_block = 32
blocks_per_grid = 2
# Launch the kernel
add_arrays_gpu[blocks_per_grid, threads_per_block](a, b, result)

# Print result
print("Array a:", a)
print("Array b:", b)
print("Result :", result)


Array a: [1. 2. 3. 4. 5.]
Array b: [5. 4. 3. 2. 1.]
Result : [6. 6. 6. 6. 6.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:940: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
